In [46]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import sys

# Change root directory to the repo root (Jupyter: __file__ is not defined)
def find_repo_root(start=Path.cwd()):
	for p in [start] + list(start.parents):
		if (p / 'Code').exists() or (p / '.git').exists() or (p / 'local_repo').exists():
			return p
	return start

repo_root = find_repo_root()
data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'

os.chdir(repo_root)
sys.path.append(str(repo_root / 'Code' / 'tools'))

# import local toolkit (try normal import first, fall back to loading from file)
try:
    import inequality_analyzers as inqA
    import local_utility_functions as luf
except Exception:
	import importlib.util
	toolkit_path = repo_root / 'Code' / 'tools' / 'inequality_analyzers.py'
	if toolkit_path.exists():
		spec = importlib.util.spec_from_file_location("inequality_analyzers", str(toolkit_path))
		stk = importlib.util.module_from_spec(spec)
		spec.loader.exec_module(stk)
	else:
		raise

In [47]:
df = pd.read_csv(data_root / 'CBOS_data_reweighted.csv')

/var/folders/y8/4_9g68pj7k136q2yypgp5ysc0000gn/T/ipykernel_40694/3946849005.py:1: DtypeWarning: Columns (15,16,21,27,28,29,33,34,35,40,41,45,46,47,79,104,105,106) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_root / 'CBOS_data_reweighted.csv')


In [48]:

# Join deflation columns to both datasets.
# We create deflator factor for each year. We collect them in 4 columns each for one base year (1986, 1990, 2017, 2023).
# Since we have monthly inflation data, we assmue that the inflation rate for the whole year is the same as the inflation rate in January of that year.

# Load monthly inflation data
# One column: inflation_index containing month to month inflation index (1.02 means 2% inflation compared to the previous month) indexed by date (first day of each month)

# data_root = repo_root.parent.parent / 'Data' / 'CBOS ready'
# → data_root.parent = repo_root.parent.parent / 'Data'
# → Auxiliary data lives at data_root.parent / 'Auxiliary data'
inflation_data = pd.read_csv(data_root.parent / 'Auxiliary data' / 'inflation_data_cleaned.csv', index_col=0, parse_dates=True)
inflation_data = inflation_data.sort_index()

# Data starts in 1982-01-01, so we can remove all rows before 1986-01-01 (the first base year)
inflation_data = inflation_data[inflation_data.index >= '1986-01-01']

print(inflation_data.head())
# First value is 102.0. -> We have to divide the index by 100 to get the actual inflation index (1.02 means 2% inflation compared to the previous month)
inflation_data['inflation_index'] = inflation_data['inflation_index'] / 100

col_mapping = {
    1986: 'deflator_1986',
    1990: 'deflator_1990',
    2017: 'deflator_2017',
    2023: 'deflator_2023'
}


            inflation_index
1986-01-01            102.0
1986-02-01            101.5
1986-03-01            102.7
1986-04-01            103.0
1986-05-01            101.6


In [49]:

# ── Step 1: Build cumulative monthly price-level index ────────────────────────
#
# inflation_index[t] = P(t) / P(t-1)  (month-to-month ratio)
# cumprod gives an absolute (arbitrarily-scaled) price level for every month.
# Only ratios between entries matter.

price_level = inflation_data['inflation_index'].cumprod()   # Timestamp → price level

# ── Step 2: Annual average price level (for yearly deflators) ─────────────────
#
# For yearly analyses the representative price level is the MEAN of all 12
# monthly price levels — not just January.  Using January alone would be
# biased, especially during hyperinflation years (1989-1991).

annual_avg_price = price_level.groupby(price_level.index.year).mean()  # int year → price level

# Monthly price level re-indexed by Period for robust matching
pl_monthly = price_level.copy()
pl_monthly.index = pl_monthly.index.to_period('M')   # Period(M) → price level

# Survey date as PeriodIndex (matches pl_monthly)
survey_periods = pd.to_datetime(df['survey_date']).dt.to_period('M')

# ── Step 3: Compute and attach 8 deflator columns ─────────────────────────────
#
# For each of the 4 base years we add TWO columns:
#   deflator_YYYY   — yearly  (annual-average price level denominator)
#                     → multiply a yearly-aggregated monetary value to express it
#                       in Jan-YYYY money
#   deflator_YYYY_m — monthly (exact survey-month price level denominator)
#                     → multiply an individual survey-month monetary value to
#                       express it in Jan-YYYY money
#
# Formula:  deflator = P(Jan base_year) / P(source_period)
# Usage:    value_in_base_money = value_in_source_money × deflator

for base_year, col_base in col_mapping.items():
    base_ts = pd.Timestamp(f'{base_year}-01-01')
    if base_ts not in price_level.index:
        print(f"[!] No price data for {base_year}-01 — skipping."); continue

    base_p = price_level[base_ts]

    # Yearly deflator: P(Jan base) / annual_avg_P(survey_year)
    yearly_defl = base_p / annual_avg_price          # Series[int year → deflator]
    df[col_base] = df['survey_year'].map(yearly_defl)

    # Monthly deflator: P(Jan base) / P(survey_month)
    monthly_defl = base_p / pl_monthly               # Series[Period M → deflator]
    df[f'{col_base}_m'] = survey_periods.map(monthly_defl)

# ── Step 4: Verification ──────────────────────────────────────────────────────
defl_y_cols = list(col_mapping.values())
defl_m_cols = [c + '_m' for c in defl_y_cols]

print("=== Yearly deflators (annual-average price level): 100 PLN → PLN in base-year money ===")
sample_y = (
    df[['survey_year'] + defl_y_cols]
    .drop_duplicates('survey_year')
    .sort_values('survey_year')
    .set_index('survey_year')
)
print((sample_y * 100).round(2).to_string())

print("\n=== Monthly deflators (exact survey month): first 12 unique months ===")
sample_m = (
    df[['survey_date'] + defl_m_cols]
    .drop_duplicates('survey_date')
    .sort_values('survey_date')
    .head(12)
)
print((sample_m.set_index('survey_date') * 100).round(2).to_string())

# Key sanity checks (base = 2017)
print("\nSanity checks (base = 2017):")
avail_years = sorted(sample_y.index.tolist())
earliest, latest = avail_years[0], avail_years[-1]
checks = [
    (earliest, ">",  f"{earliest} pre-hyperinflation → >> 100 in 2017 PLN"),
    (2017,     "≈1", "2017 yearly ≈ ~100 in 2017 PLN (slightly < 1 because annual avg > Jan price)"),
    (latest,   "<",  f"{latest} prices higher than 2017 → < 100 in 2017 PLN"),
]
for yr, direction, note in checks:
    if yr not in sample_y.index: print(f"  [!] {yr} not in data"); continue
    v = 100 * sample_y.loc[yr, 'deflator_2017']
    ok = (direction == ">" and v > 100) or (direction == "<" and v < 100) or (direction == "≈1" and 85 < v < 115)
    print(f"  {'✓' if ok else '✗'}  100 PLN ({yr}) = {v:>10,.2f} in 2017 money  [{note}]")

# Monthly check: Jan 2017 should be exactly 100 with monthly deflator (base=Jan 2017)
jan2017_m = df.loc[df['survey_date'] == '2017-01-01', 'deflator_2017_m']
if not jan2017_m.empty:
    v = 100 * jan2017_m.iloc[0]
    print(f"  {'✓' if abs(v - 100) < 1e-6 else '✗'}  Monthly: 100 PLN (Jan 2017) = {v:.6f} in 2017 money  [exact base month → must be 100.000000]")

print(f"\nNaN counts:\n  yearly  {df[defl_y_cols].isna().sum().to_dict()}")
print(f"  monthly {df[defl_m_cols].isna().sum().to_dict()}")
print(f"\nNew columns: {defl_y_cols + defl_m_cols}")


=== Yearly deflators (annual-average price level): 100 PLN → PLN in base-year money ===
             deflator_1986  deflator_1990  deflator_2017  deflator_2023
survey_year                                                            
1990                  1.90          65.94        1571.85        2220.56
1991                  1.08          37.37         890.85        1258.52
1992                  0.74          25.72         612.98         865.97
1993                  0.54          18.79         447.76         632.55
1994                  0.41          14.10         336.10         474.82
1995                  0.32          11.01         262.42         370.72
1996                  0.26           9.19         219.01         309.39
1997                  0.23           7.98         190.30         268.85
1998                  0.21           7.15         170.36         240.67
1999                  0.19           6.66         158.76         224.29
2000                  0.17           6.05       

In [50]:
groups_dict = {"teryt_id_VOIV_500": {},
               "teryt_id_VOIV_100_500": {},
               "teryt_id_VOIV_100": {}}

for year in df['survey_year'].unique():
    for col in groups_dict.keys():
        temp = df.loc[df['survey_year'] == year, [col, str('G') + col[8:]]].drop_duplicates()
        groups_dict[col][int(year)] = temp.set_index(str('G') + col[8:]).to_dict()

# Dump as JSON for later use in the paper (e.g. to label figures with group names)
import json
json_outpath = data_root.parent / "replication_package" / "CBOS" / 'CBOS_group_mappings.json'
with open(json_outpath, 'w') as f:
    json.dump(groups_dict, f, indent=4)
print(f"Saved group mappings → {json_outpath}")


Saved group mappings → /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/replication_package/CBOS/CBOS_group_mappings.json


In [51]:
# Save as CSV for later use in the paper (e.g. to label figures with group names)
df.to_csv(data_root / 'CBOS_data_reweighted.csv', index=False)

# Remove the very heavy column for a lite version of the dataset (if needed)
df_lite = df.copy()
df_lite.drop(columns=['teryt_id_VOIV_500', 'teryt_id_VOIV_100_500', 'teryt_id_VOIV_100'], inplace=True)
df_lite.to_csv(data_root.parent / "replication_package" / "CBOS" / 'CBOS_survey.csv', index=False)
print(f"Saved lite version (without heavy VOIV columns) → {data_root.parent / 'replication_package' / 'CBOS' / 'CBOS_survey.csv'}")

Saved lite version (without heavy VOIV columns) → /Users/jedrek/Documents/Studium Volkswirschaftslehre/3. Semester/Long-run dynamics of wealth inequalities/Paper/Data/replication_package/CBOS/CBOS_survey_lite.csv


In [55]:

# ── ANALYSIS: Structure of df_lite and size optimization ──────────────────────

print("="*80)
print("STRUCTURE ANALYSIS OF df_lite")
print("="*80)

print("\n1. COLUMN DATA TYPES AND SIZES:")
print("-" * 80)
dtypes_info = pd.DataFrame({
    'Column': df_lite.columns,
    'Type': df_lite.dtypes,
    'Non-Null': df_lite.count(),
    'Null%': (df_lite.isna().sum() / len(df_lite) * 100).round(2),
    'Size (MB)': [df_lite[col].memory_usage(deep=True) / 1024**2 for col in df_lite.columns]
})
print(dtypes_info.to_string(index=False))

total_mb_original = df_lite.memory_usage(deep=True).sum() / 1024**2
print(f"\nTotal size (original): {total_mb_original:.2f} MB")

print("\n2. COLUMNS THAT ARE STRINGS (CODES - KEEP AS-IS):")
print("-" * 80)
string_cols = df_lite.select_dtypes(include=['object']).columns.tolist()
for col in string_cols:
    n_unique = df_lite[col].nunique()
    sample = df_lite[col].dropna().unique()[:3]
    print(f"  • {col}: {n_unique} unique values | Sample: {sample}")
    print(f"    → KEEP AS STRING (identifier codes)")

print("\n3. NUMERIC COLUMNS - OPTIMIZATION OPPORTUNITIES:")
print("-" * 80)
numeric_cols = df_lite.select_dtypes(include=['float64', 'int64']).columns.tolist()
for col in numeric_cols:
    col_min = df_lite[col].min()
    col_max = df_lite[col].max()
    has_decimals = not df_lite[col].dropna().apply(lambda x: x == int(x)).all()
    print(f"  • {col}: [{col_min:.2f}, {col_max:.2f}] | Decimals: {has_decimals}")

print("\n" + "="*80)
print("SIZE OPTIMIZATION STRATEGY")
print("="*80)
print("""
Strategy:
  1. Keep all string columns as-is (they are codes: teryt_*, respondent_id, etc.)
  2. Convert float64 → float32 for deflator columns (saves ~50% memory space)
  3. Convert other float64 → float32 across the board
  4. Round deflators to 6 decimals (adequate precision for monetary values)
  5. Save as compressed CSV (.gz format ~70-80% compression vs plain text)
""")

# ── Optimize df_lite and save ──────────────────────────────────────────────────

df_opt = df_lite.copy()

# Round deflator columns to 6 decimals, then convert to float32
deflator_cols = [c for c in df_opt.columns if 'deflator' in c]
for col in deflator_cols:
    df_opt[col] = df_opt[col].round(6).astype('float32')

# Convert all remaining float64 columns to float32
for col in df_opt.select_dtypes(include=['float64']).columns:
    if col not in deflator_cols:
        df_opt[col] = df_opt[col].astype('float32')

print("\n4. OPTIMIZED COLUMN TYPES:")
print("-" * 80)
dtypes_opt = pd.DataFrame({
    'Column': df_opt.columns,
    'Type': df_opt.dtypes,
    'Size (MB)': [df_opt[col].memory_usage(deep=True) / 1024**2 for col in df_opt.columns]
})
print(dtypes_opt.to_string(index=False))

total_mb_optimized = df_opt.memory_usage(deep=True).sum() / 1024**2
print(f"\nTotal size (optimized in RAM): {total_mb_optimized:.2f} MB")
print(f"Memory reduction: {(1 - total_mb_optimized/total_mb_original)*100:.1f}%")

# Save as compressed CSV (gzip format)
csv_gz_path = data_root.parent / "replication_package" / "CBOS" / 'CBOS_survey.csv.gz'
df_opt.to_csv(csv_gz_path, index=False, compression='gzip')
gz_size_mb = csv_gz_path.stat().st_size / 1024**2
print(f"\n✓ Saved (CSV + gzip compression): {csv_gz_path}")
print(f"  File size: {gz_size_mb:.2f} MB")

# Also save plain CSV for compatibility  
csv_path = data_root.parent / "replication_package" / "CBOS" / 'CBOS_survey.csv'
df_opt.to_csv(csv_path, index=False)
csv_size_mb = csv_path.stat().st_size / 1024**2
print(f"\n✓ Saved (CSV uncompressed): {csv_path}")
print(f"  File size: {csv_size_mb:.2f} MB")

print(f"\n{'='*80}")
print("FILE SIZE COMPARISON:")
print(f"{'='*80}")
print(f"  In-memory (original floats):  {total_mb_original:.2f} MB")
print(f"  In-memory (optimized):        {total_mb_optimized:.2f} MB  ({(1 - total_mb_optimized/total_mb_original)*100:.1f}% smaller)")
print(f"  CSV uncompressed:             {csv_size_mb:.2f} MB")
print(f"  CSV + gzip compression:       {gz_size_mb:.2f} MB  ← RECOMMENDED ({(1 - gz_size_mb/csv_size_mb)*100:.1f}% smaller than CSV)")
print(f"\nRecommendation:")
print(f"  • Use CBOS_survey.csv.gz for storage/archival (~{gz_size_mb:.1f} MB)")
print(f"  • Use CBOS_survey.csv for direct analysis (~{csv_size_mb:.1f} MB, easily readable)")


STRUCTURE ANALYSIS OF df_lite

1. COLUMN DATA TYPES AND SIZES:
--------------------------------------------------------------------------------
                     Column    Type  Non-Null  Null%  Size (MB)
                 Unnamed: 0   int64    355337   0.00   2.711132
                     org_id float64    355337   0.00   2.711132
                survey_file  object    355337   0.00  23.257257
                survey_year   int64    355337   0.00   2.711132
               survey_month   int64    355337   0.00   2.711132
                        age float64    355337   0.00   2.711132
                  year_born float64    355337   0.00   2.711132
                        sex float64    355337   0.00   2.711132
                      sex_L  object    355337   0.00  25.160984
                  city_size float64    355337   0.00   2.711132
                city_size_L  object    355337   0.00  36.612517
                  education float64    355337   0.00   2.711132
                educatio